In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
import logging
print('pandas:', pd.__version__, ' polars:', pl.__version__)
import math


In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

logger = logging.getLogger("test_preprocess")
logger.addHandler(logging.NullHandler())

def _clean_column_names_pandas(df):
    original_columns = df.columns.tolist()
    cleaned_columns = []
    for i, col in enumerate(original_columns):
        if pd.isna(col) or col == "":
            cleaned_columns.append(f"column_{i}")
        else:
            cleaned_columns.append(str(col))
    return cleaned_columns, original_columns

def _clean_column_names_polars(df):
    original_columns = df.columns
    cleaned_columns = []
    for i, col in enumerate(original_columns):
        if col is None or col == "":
            cleaned_columns.append(f"column_{i}")
        elif isinstance(col, float) and math.isnan(col):
            cleaned_columns.append(f"column_{i}")
        else:
            cleaned_columns.append(str(col))
    return cleaned_columns, original_columns

# --- preprocess_attrs_registry ---
FIX_PREPROCESS_ATTRS_REGISTRY_CLEAN_COLUMN_NAMES = _clean_column_names_pandas
FIX_PREPROCESS_ATTRS_REGISTRY_CLEAN_COLUMN_NAMES_PL = _clean_column_names_polars

# --- preprocess_columns_tolist ---

# --- preprocess_shape_rename ---
FIX_PREPROCESS_SHAPE_RENAME_CLEAN_COLUMN_NAMES = _clean_column_names_pandas
FIX_PREPROCESS_SHAPE_RENAME_CLEAN_COLUMN_NAMES_PL = _clean_column_names_polars

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_preprocess_attrs_registry(clean_column_names):
    def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
        logger.info(
            f"Starting DataFrame preprocessing: {df.shape[0]} rows, {df.shape[1]} columns"
        )

        cleaned_columns, original_columns = clean_column_names(df)

        processed_df = df.copy()
        processed_df.columns = cleaned_columns

        processed_df.attrs["original_columns"] = original_columns
        processed_df.attrs["column_mapping"] = dict(zip(cleaned_columns, original_columns))

        logger.info("DataFrame preprocessing completed successfully")
        logger.debug(
            f"Column mapping created: {len(processed_df.attrs['column_mapping'])} entries"
        )

        return processed_df

    def get_original_column_name(df: pd.DataFrame, cleaned_name: str) -> str:
        if hasattr(df, "attrs") and "column_mapping" in df.attrs:
            original_name = df.attrs["column_mapping"].get(cleaned_name, cleaned_name)
            if original_name != cleaned_name:
                logger.debug(
                    f"Retrieved original column name: '{cleaned_name}' -> '{original_name}'"
                )
            return original_name

        logger.debug(f"No column mapping found, returning cleaned name: '{cleaned_name}'")
        return cleaned_name
    return get_original_column_name

def before_preprocess_columns_tolist():
    def clean_column_names(df: pd.DataFrame):
        original_columns = df.columns.tolist()
        cleaned_columns = []

        for i, col in enumerate(original_columns):
            if pd.isna(col) or col == "":
                cleaned_columns.append(f"column_{i}")
            else:
                cleaned_columns.append(str(col))

        return cleaned_columns, original_columns
    return clean_column_names

def before_preprocess_shape_rename(clean_column_names):
    def preprocess_dataframe(df: pd.DataFrame) -> pd.DataFrame:
        logger.info(
            f"Starting DataFrame preprocessing: {df.shape[0]} rows, {df.shape[1]} columns"
        )

        cleaned_columns, original_columns = clean_column_names(df)

        processed_df = df.copy()
        processed_df.columns = cleaned_columns

        processed_df.attrs["original_columns"] = original_columns

        return processed_df
    return preprocess_dataframe

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_preprocess_attrs_registry(clean_column_names):
    def preprocess_dataframe(df: pl.DataFrame) -> pl.DataFrame:
        logger.info(
            f"Starting DataFrame preprocessing: {df.shape[0]} rows, {df.shape[1]} columns"
        )

        cleaned_columns, original_columns = clean_column_names(df)

        processed_df = df.clone()
        processed_df.columns = cleaned_columns

        try:
            processed_df.attrs["original_columns"] = original_columns
            processed_df.attrs["column_mapping"] = dict(zip(cleaned_columns, original_columns))
        except Exception:
            pass

        logger.info("DataFrame preprocessing completed successfully")
        logger.debug(
            f"Column mapping created: {len(dict(zip(cleaned_columns, original_columns)))} entries"
        )

        return processed_df

    def get_original_column_name(df: pl.DataFrame, cleaned_name: str) -> str:
        if hasattr(df, "attrs") and "column_mapping" in df.attrs:
            original_name = df.attrs["column_mapping"].get(cleaned_name, cleaned_name)
            if original_name != cleaned_name:
                logger.debug(
                    f"Retrieved original column name: '{cleaned_name}' -> '{original_name}'"
                )
            return original_name

        logger.debug(f"No column mapping found, returning cleaned name: '{cleaned_name}'")
        return cleaned_name
    return get_original_column_name

def gen_preprocess_columns_tolist():
    def clean_column_names(df: pl.DataFrame):
        original_columns = df.columns
        cleaned_columns = []

        for i, col in enumerate(original_columns):
            if col is None or col == "":
                cleaned_columns.append(f"column_{i}")
            elif isinstance(col, float) and math.isnan(col):
                cleaned_columns.append(f"column_{i}")
            else:
                cleaned_columns.append(str(col))

        return cleaned_columns, original_columns
    return clean_column_names

def gen_preprocess_shape_rename(clean_column_names):
    def preprocess_dataframe(df: pl.DataFrame) -> pl.DataFrame:
        logger.info(
            f"Starting DataFrame preprocessing: {df.shape[0]} rows, {df.shape[1]} columns"
        )

        cleaned_columns, original_columns = clean_column_names(df)

        processed_df = df.clone()
        processed_df = processed_df.rename(
            dict(zip(processed_df.columns, cleaned_columns))
        )

        try:
            processed_df.attrs["original_columns"] = original_columns
        except Exception:
            pass

        return processed_df
    return preprocess_dataframe

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: preprocess_attrs_registry ===
print("⚠️ L1 skip preprocess_attrs_registry: evaluation-invalid — isolated source exposes only the lookup closure")
print("⚠️ L2 skip preprocess_attrs_registry: evaluation-invalid — the preprocessing closure that creates the mapping is not observable")
print("⚠️ L3 skip preprocess_attrs_registry: evaluation-invalid — complete paired behaviour requires the enclosing function context")
